In [3]:
import requests
from bs4 import BeautifulSoup
import json
import re

url = "https://sweb.hku.hk/ccacad/ccc_appl/enrol_stat.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

enrol_data = {}

# Find all table rows
for tr in soup.find_all("tr"):
    tds = [td.get_text().strip() for td in tr.find_all("td")]
    # Check if row is a valid course row (starts with CC code like CCST9003)
    if len(tds) >= 6 and re.match(r"^CC[A-Z]{2}\d{4}$", tds[0]):
        code, title, subclass, quota, vacancies, waiting = tds[:6]

        try:
            quota = int(quota)
            vacancies = int(vacancies)
            waiting = int(waiting)
        except ValueError:
            continue

        if code not in enrol_data:
            enrol_data[code] = {
                "total_quota": 0,
                "total_vacancies": 0,
                "total_waiting": 0,
                "subclasses": []
            }

        enrol_data[code]["total_quota"] += quota
        enrol_data[code]["total_vacancies"] += vacancies
        enrol_data[code]["total_waiting"] += waiting
        enrol_data[code]["subclasses"].append(subclass)

with open("enrol-info.json", "w") as f:
    json.dump(enrol_data, f, indent=4)

print(f"Successfully scraped enrollment stats for {len(enrol_data)} courses!")

Successfully scraped enrollment stats for 201 courses!
